In [0]:
%run ./00_config

In [0]:
%run ./02_core_functions

# 03_main_hsar - daily
For every active **HSAR** row in the control table:

1. **Copy** zip files from the source to `<zip_tgt_path>/<mid_date>/`
2. **Unzip** them into `<unzip_tgt_path>/<mid_date>/<zip_name>/`
3. **Load** each affected `<mid_date>` folder with Auto Loader (one row per JSON file, full content kept, tagged with its `zip_file`)

**First run:** every zip at the source is loaded.
**Later runs (daily):** only *new* zips (not yet in `raw/`), so a day that was missed is caught up automatically. The preview below checks the **END date** of the zips (the daily file) and warns if there is none for today. Set `is_full_load = 1` in the control table to reload everything.

If a zip fails to unzip, or its folder fails to load, the raw copy is removed (and the folder's partial rows and checkpoint cleared) so the next run retries it cleanly.

In [0]:
run_id          = str(uuid.uuid4())
control_entries = get_active_control_entries("HSAR")
ctl_by_sno      = {c["s_no"]: c for c in control_entries}
log_records     = []

print(f"run_id = {run_id}")
print(f"{len(control_entries)} active HSAR control row(s)")

### Preview - which daily files will be loaded?

In [0]:
today = datetime.now().strftime("%Y%m%d")
for ctl in control_entries:
    all_zips = list_zip_files(ctl["zip_source_path"])
    zips     = select_zips_to_copy(all_zips, ctl)
    print(f"s_no={ctl['s_no']}: {len(zips)} new zip(s); END dates to load: {zip_end_dates(zips) or '-'}")
    if today not in zip_end_dates(all_zips):
        print(f"  WARNING: no zip with END date {today} (today) at the source yet")

### Stage 1 + 2 - Copy and Unzip

In [0]:
copy_and_unzip(run_id, control_entries, log_records)

### Stage 3 - Load dated folders (Auto Loader)

In [0]:
# dated folders that received zips in this run: {(s_no, yyyymmdd): [unzip records]}
folders = {}
for r in log_records:
    if r["stage"] == "UNZIP" and r["status"] == "SUCCESS":
        folders.setdefault((r["s_no"], zip_folder_date(r["file_name"])), []).append(r)
print(f"{len(folders)} dated folder(s) to load")

load_records = []
for (s_no, dated_folder), unzip_recs in sorted(folders.items()):
    ctl    = ctl_by_sno[s_no]
    target = f"{ctl['tgt_catalog']}.{ctl['tgt_schema']}.{ctl['tgt_table']}"
    rec = {"s_no": s_no, "stage": "HSAR_LOAD", "source": ctl["src_system"], "file_name": dated_folder,
           "source_path": f"{ctl['unzip_tgt_path'].rstrip('/')}/{dated_folder}/", "target_path": target,
           "start_time": datetime.now()}
    try:
        if ctl["is_full_load"] == 1:
            hsar_reset_folder(ctl, dated_folder)
        n_files = hsar_load_folder(ctl, dated_folder)
        rec.update(status="SUCCESS", message=f"folder has {n_files} file(s) in the target after loading {len(unzip_recs)} new zip(s)")
    except Exception as e:
        rec.update(status="FAILED", message=str(e)[:4000])
        try:
            hsar_reset_folder(ctl, dated_folder)                  # clean slate for the retry
        except Exception as reset_err:
            print(f"WARNING: could not reset {dated_folder}: {reset_err}")
        for u in unzip_recs:
            discard_raw_copy(u["source_path"])                    # zips are copied/unzipped/loaded again next run
    rec["end_time"] = datetime.now()
    load_records.append(rec)
    print(f"  [HSAR:{rec['status']}] {dated_folder}: {rec['message'][:300]}")

write_logs(load_records, run_id)
print(f"Logged {len(load_records)} HSAR_LOAD record(s) to {log_table}")

### Run summary

In [0]:
display(spark.table(log_table).filter(F.col("run_id") == run_id).orderBy("start_time"))

In [0]:
%sql
select * from iics.casemix.hsar_json_test
-- where zip_file='carereport_hsar_1007320371375345779_20250309_20250328_20261022_000002.zip'
order by file_id desc

In [0]:
# Fail the notebook / job if anything failed (details are in the log table for this run_id)
failed = sum(1 for r in log_records + load_records if r["status"] == "FAILED")
if failed > 0:
    raise Exception(f"{failed} step(s) failed. See {log_table} where run_id = '{run_id}'")